In [1]:
import pandas as pd

BASE = "s3://customer-dc-grai-matter-prod"

path = f"{BASE}/grai_matter_fct_icd_codes/"
cols = ["PRIMARY_PATIENT_IDENTIFIER", "DH_ENCOUNTER_ID", "ICD10_CODE", "ICD10_DT"]

try:
    icd = pd.read_parquet(
        path,
        columns=cols,
        dtype_backend="pyarrow"
    )
except Exception:
    fs = s3fs.S3FileSystem()
    icd = pd.read_parquet(
        path,
        columns=cols,
        engine="pyarrow",
        filesystem=fs
    )

# Standardize column names
icd = icd.rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER": "patient_id",
    "DH_ENCOUNTER_ID": "encounter_id",
    "ICD10_CODE": "icd10_code",
    "ICD10_DT": "icd10_dt"
})

# Enforce string dtypes for IDs/codes
for c in ["patient_id", "encounter_id", "icd10_code"]:
    if c in icd.columns:
        icd[c] = icd[c].astype("string[pyarrow]")

# Normalize date/time column
icd["icd10_dt"] = pd.to_datetime(icd["icd10_dt"], errors="coerce")

# Drop empty keys/codes/dates as needed and deduplicate
icd = (
    icd
    .dropna(subset=["patient_id", "encounter_id", "icd10_code"])  
    .drop_duplicates()
)

print(icd.head(5))

                                          patient_id  \
0  ea2367c08d1fd69647a8c47fbc80c9950152bca5998cb0...   
1  a4574c6ef828d06d4a0ab428ed4ff6ac9e25a1173528e9...   
2  0bde64480391b725b63169926bb351e613741b47987686...   
3  5087df907489e252c3737ec5764873f7e2b6fffde515f5...   
4  e0326066b332401466467787242b546500701518d44f34...   

                                        encounter_id icd10_code  \
0  9a3821696c8d22a21042b13f89b61e29fb7d4ace0a10aa...    C50.919   
1  6a0fd6b5db0a6ce4675e50865c15de77e5145f29785ef2...     O09.93   
2  7cbe2929edd0bcf7c95062fe04f357967c047857638a49...      F41.1   
3  577b3eae53280f5cca19f96f5c3a15c28d79579a48d91f...    Z86.010   
4  fd559f7cb89663295b2c596512b2dfbf3255881b208b11...        R17   

                   icd10_dt  
0                       NaT  
1                       NaT  
2 2022-12-16 00:00:00+00:00  
3 2023-01-05 00:00:00+00:00  
4                       NaT  


In [2]:
# Clean and uppercase (idempotent if already clean)
icd["icd10_code_clean"] = (
    icd["icd10_code"]
    .astype("string[pyarrow]")
    .str.strip()
    .str.upper()
)

# Derive 3-char prefix (category): remove dots and non-alnum, then take first 3
icd["icd10_prefix3"] = (
    icd["icd10_code_clean"]
    .str.replace(r"[^A-Z0-9]", "", regex=True)
    .str[:3]
)

# Build the official event-level table and store it in patient_features
patient_features = (
    icd.loc[:, ["patient_id", "encounter_id", "icd10_prefix3", "icd10_dt"]]
       .rename(columns={
           "icd10_prefix3": "icd_prefix3",
           "icd10_dt": "icd_dt"
       })
       .dropna(subset=["patient_id", "encounter_id", "icd_prefix3"])
       .drop_duplicates()
       .reset_index(drop=True)
)

# Enforce consistent dtypes
for c in ["patient_id", "encounter_id", "icd_prefix3"]:
    patient_features[c] = patient_features[c].astype("string[pyarrow]")

print(patient_features.head(5))

                                          patient_id  \
0  ea2367c08d1fd69647a8c47fbc80c9950152bca5998cb0...   
1  a4574c6ef828d06d4a0ab428ed4ff6ac9e25a1173528e9...   
2  0bde64480391b725b63169926bb351e613741b47987686...   
3  5087df907489e252c3737ec5764873f7e2b6fffde515f5...   
4  e0326066b332401466467787242b546500701518d44f34...   

                                        encounter_id icd_prefix3  \
0  9a3821696c8d22a21042b13f89b61e29fb7d4ace0a10aa...         C50   
1  6a0fd6b5db0a6ce4675e50865c15de77e5145f29785ef2...         O09   
2  7cbe2929edd0bcf7c95062fe04f357967c047857638a49...         F41   
3  577b3eae53280f5cca19f96f5c3a15c28d79579a48d91f...         Z86   
4  fd559f7cb89663295b2c596512b2dfbf3255881b208b11...         R17   

                     icd_dt  
0                       NaT  
1                       NaT  
2 2022-12-16 00:00:00+00:00  
3 2023-01-05 00:00:00+00:00  
4                       NaT  


In [3]:
# aggregate to encounter-level
encounter_level = (
    patient_features
    .groupby(["patient_id","encounter_id"], as_index=False)
    .agg(
        # comma-separated sorted unique prefixes 
        icd_prefix3_unique_str=("icd_prefix3", lambda s: ",".join(sorted(set(s.dropna())))),
        # unique count of prefixes
        n_icd_prefix3=("icd_prefix3", lambda s: s.dropna().nunique()),
    )
)

# enforce dtypes
for c in ["patient_id","encounter_id","icd_prefix3_unique_str"]:
    encounter_level[c] = encounter_level[c].astype("string[pyarrow]")
encounter_level["n_icd_prefix3"] = encounter_level["n_icd_prefix3"].astype("Int64")

# write back to official store
patient_features = encounter_level

print(patient_features.head(5))

                                          patient_id  \
0  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
1  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
2  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
3  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
4  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   

                                        encounter_id  \
0  0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...   
1  170bb050a38ac8bc59d1f95c911ec58a45eb4c9033ada9...   
2  90568f52952529790fc1ee3205e45bc7f7dd57eb2af361...   
3  aa7ec13bccf95f1278dfe60e4006dfe45ce3fbdded2af7...   
4  b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...   

                              icd_prefix3_unique_str  n_icd_prefix3  
0                            E11,E86,F31,I50,J44,N30              6  
1                                                R53              1  
2                                            E87,R53              2  
3                                             

In [4]:
enc_path = f"{BASE}/grai_matter_fct_encounters/"
enc_cols = [
    "PRIMARY_PATIENT_IDENTIFIER",
    "DH_ENCOUNTER_ID",
    "ENCOUNTER_START_DT",
    "ENCOUNTER_END_DT",
    "DH_ENCOUNTER_SETTING",
    "DH_ENCOUNTER_MEDICAL_SERVICE",
]

try:
    enc = pd.read_parquet(enc_path, columns=enc_cols, dtype_backend="pyarrow")
except Exception:
    fs = s3fs.S3FileSystem()
    enc = pd.read_parquet(enc_path, columns=enc_cols, engine="pyarrow", filesystem=fs)

# --- Standardize column names ---
enc = enc.rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER": "patient_id",
    "DH_ENCOUNTER_ID": "encounter_id",
    "ENCOUNTER_START_DT": "encounter_start_dt",
    "ENCOUNTER_END_DT": "encounter_end_dt",
    "DH_ENCOUNTER_SETTING": "encounter_setting",
    "DH_ENCOUNTER_MEDICAL_SERVICE": "encounter_medical_service",
})

# --- Enforce dtypes ---
for c in ["patient_id", "encounter_id", "encounter_setting", "encounter_medical_service"]:
    enc[c] = enc[c].astype("string[pyarrow]")
enc["encounter_start_dt"] = pd.to_datetime(enc["encounter_start_dt"], errors="coerce")
enc["encounter_end_dt"]   = pd.to_datetime(enc["encounter_end_dt"], errors="coerce")

# --- Keep one row per (patient_id, encounter_id) ---
enc = (
    enc
    .dropna(subset=["patient_id", "encounter_id"])
    .drop_duplicates(subset=["patient_id", "encounter_id"])
    .loc[:, ["patient_id","encounter_id","encounter_start_dt","encounter_end_dt",
             "encounter_setting","encounter_medical_service"]]
)

# --- Merge into official store (left join on patient_id + encounter_id) ---
patient_features = (
    patient_features
    .merge(enc, on=["patient_id","encounter_id"], how="left")
)

# --- compute duration in days 
patient_features["encounter_duration_days"] = (
    (patient_features["encounter_end_dt"] - patient_features["encounter_start_dt"])
    .dt.total_seconds() / (3600.0 * 24)
)

patient_features["encounter_duration_days"] = patient_features["encounter_duration_days"].astype("Float64")

# --- place the new column immediately after encounter_end_dt ---
if "encounter_end_dt" not in patient_features.columns:
    raise KeyError("`encounter_end_dt` is missing after merge; cannot place duration column.")
cols = list(patient_features.columns)
end_idx = cols.index("encounter_end_dt")  # position of end time
# build new order by inserting the brand-new column right after end time
new_cols = cols[:end_idx + 1] + ["encounter_duration_days"] + cols[end_idx + 1:-1]
patient_features = patient_features.reindex(columns=new_cols)

print(patient_features.head(5))

                                          patient_id  \
0  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
1  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
2  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
3  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
4  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   

                                        encounter_id  \
0  0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...   
1  170bb050a38ac8bc59d1f95c911ec58a45eb4c9033ada9...   
2  90568f52952529790fc1ee3205e45bc7f7dd57eb2af361...   
3  aa7ec13bccf95f1278dfe60e4006dfe45ce3fbdded2af7...   
4  b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...   

                              icd_prefix3_unique_str  n_icd_prefix3  \
0                            E11,E86,F31,I50,J44,N30              6   
1                                                R53              1   
2                                            E87,R53              2   
3                                         

In [5]:
adt_path = f"{BASE}/grai_matter_dim_adt/"
adt_cols = [
    "PRIMARY_PATIENT_IDENTIFIER",
    "DH_ENCOUNTER_ID",
    "EVENT_TYPE",
    "EFFECTIVE_TIME",
]

try:
    adt = pd.read_parquet(adt_path, columns=adt_cols, dtype_backend="pyarrow")
except Exception:
    fs = s3fs.S3FileSystem()
    adt = pd.read_parquet(adt_path, columns=adt_cols, engine="pyarrow", filesystem=fs)

# --- standardize column names ---
adt = adt.rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER": "patient_id",
    "DH_ENCOUNTER_ID": "encounter_id",
    "EVENT_TYPE": "adt_event_type",
    "EFFECTIVE_TIME": "adt_effective_time",
})

# --- dtypes & basic cleanup ---
adt["patient_id"] = adt["patient_id"].astype("string[pyarrow]")
adt["encounter_id"] = adt["encounter_id"].astype("string[pyarrow]")
adt["adt_event_type"] = adt["adt_event_type"].astype("string[pyarrow]")
adt["adt_effective_time"] = pd.to_datetime(adt["adt_effective_time"], errors="coerce")

adt = (
    adt
    .dropna(subset=["patient_id", "encounter_id", "adt_event_type", "adt_effective_time"])
    .drop_duplicates(subset=["patient_id", "encounter_id", "adt_event_type", "adt_effective_time"])
)

# --- order events within each encounter and build sequences ---
adt_sorted = adt.sort_values(["patient_id", "encounter_id", "adt_effective_time"])

def _join_types(s: pd.Series) -> str:
    # join event types in chronological order
    return "|".join(s.astype(str))

def _join_times(s: pd.Series) -> str:
    # join timestamps in chronological order; fallback to str if tz formatting fails
    try:
        return "|".join(s.dt.strftime("%Y-%m-%d %H:%M:%S%z"))
    except Exception:
        return "|".join(s.astype(str))

adt_seq = (
    adt_sorted
    .groupby(["patient_id", "encounter_id"], as_index=False)
    .agg(
        adt_event_types_seq_str=("adt_event_type", _join_types),
        adt_event_times_seq_str=("adt_effective_time", _join_times),
    )
)

# --- enforce dtypes for outputs ---
for c in ["patient_id", "encounter_id", "adt_event_types_seq_str", "adt_event_times_seq_str"]:
    adt_seq[c] = adt_seq[c].astype("string[pyarrow]")

# --- merge into official store (encounter-level) ---
patient_features = patient_features.merge(
    adt_seq, on=["patient_id", "encounter_id"], how="inner", validate="one_to_one" 
)

print(patient_features.head(5))

                                          patient_id  \
0  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
1  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
2  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
3  00010b29ec5fd2ad40834318f6b429961059a614fbfece...   
4  000183804ffd1929b01d145a9cbf729d9c411d64675c02...   

                                        encounter_id  \
0  0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...   
1  b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...   
2  c67b56e4f422be9a8894634536e8895da82d159b2ed583...   
3  0d79127152552683ecf7ba4acd391eeaaa6e90cceffb39...   
4  affeaca0615c02f762a8404e381b3b86243ee1b7e42bf2...   

                              icd_prefix3_unique_str  n_icd_prefix3  \
0                            E11,E86,F31,I50,J44,N30              6   
1  E11,E78,E86,F17,F31,F41,F43,G70,I50,J44,K21,N3...             15   
2                                            E87,R53              2   
3                                         

In [6]:
# ---- Demographics from dim_patient ----
dem = pd.read_parquet(
    f"{BASE}/grai_matter_dim_patient/",
    columns=["PRIMARY_PATIENT_IDENTIFIER","PATIENT_BIRTH_YEAR","PATIENT_SEX","PATIENT_RACE_ETHNICITY","DECEASED_FLAG"],
    dtype_backend="pyarrow"
).rename(columns={"PRIMARY_PATIENT_IDENTIFIER":"patient_id"})

dem["PATIENT_BIRTH_YEAR"] = pd.to_numeric(dem["PATIENT_BIRTH_YEAR"], errors="coerce").astype("Int64")
dem["patient_id"] = dem["patient_id"].astype(str)
current_year = pd.Timestamp.now(tz="UTC").year
dem["age"] = (current_year - dem["PATIENT_BIRTH_YEAR"]).astype("Int64")

patient_features = patient_features.merge(dem, on="patient_id", how="left")
patient_features.head(5)

,patient_id,encounter_id,icd_prefix3_unique_str,n_icd_prefix3,encounter_start_dt,encounter_end_dt,encounter_duration_days,encounter_setting,encounter_medical_service,adt_event_types_seq_str,adt_event_times_seq_str,PATIENT_BIRTH_YEAR,PATIENT_SEX,PATIENT_RACE_ETHNICITY,DECEASED_FLAG,age
0,0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...,0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...,"E11,E86,F31,I50,J44,N30",6,2022-11-19 00:00:00+00:00,2022-11-21 03:15:00+00:00,2.135417,Observation,Nursing - medical / surgical,Admission|Census|Census|Discharge,2022-11-19 10:32:00+0000|2022-11-19 11:59:00+0...,1948,Female,"White, non-Hispanic",1,77
1,0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...,b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...,"E11,E78,E86,F17,F31,F41,F43,G70,I50,J44,K21,N3...",15,2022-11-19 00:00:00+00:00,2022-11-19 11:59:00+00:00,0.499306,Outpatient,Laboratory,Hospital Outpatient|Discharge,2022-11-19 01:25:00+0000|2022-11-19 11:59:00+0000,1948,Female,"White, non-Hispanic",1,77
2,0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...,c67b56e4f422be9a8894634536e8895da82d159b2ed583...,"E87,R53",2,2022-12-11 00:00:00+00:00,2022-12-11 11:59:00+00:00,0.499306,Outpatient,Laboratory,Hospital Outpatient|Discharge,2022-12-11 07:30:00+0000|2022-12-11 11:59:00+0000,1948,Female,"White, non-Hispanic",1,77
3,00010b29ec5fd2ad40834318f6b429961059a614fbfece...,0d79127152552683ecf7ba4acd391eeaaa6e90cceffb39...,T18,1,2022-11-10 00:00:00+00:00,2022-11-12 02:55:00+00:00,2.121528,Observation,General surgery,Admission|Transfer Out|Transfer In|Transfer Ou...,2022-11-10 07:43:00+0000|2022-11-10 11:10:00+0...,1983,Male,Hispanic,0,42
4,000183804ffd1929b01d145a9cbf729d9c411d64675c02...,affeaca0615c02f762a8404e381b3b86243ee1b7e42bf2...,"G89,K42,K66,N83,Z30,Z90,Z98",7,2022-12-16 00:00:00+00:00,2022-12-16 11:59:00+00:00,0.499306,Outpatient,General surgery,Hospital Outpatient|Discharge,2022-12-16 07:57:00+0000|2022-12-16 11:59:00+0000,1991,Female,"White, non-Hispanic",0,34


In [7]:
med_path = f"{BASE}/grai_matter_fct_medication/"
med_cols = ["PRIMARY_PATIENT_IDENTIFIER", "DH_ENCOUNTER_ID", "DH_ORDER_ID"]

try:
    med = pd.read_parquet(med_path, columns=med_cols, dtype_backend="pyarrow")
except Exception:
    fs = s3fs.S3FileSystem()
    med = pd.read_parquet(med_path, columns=med_cols, engine="pyarrow", filesystem=fs)

# --- Standardize column names ---
med = med.rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER": "patient_id",
    "DH_ENCOUNTER_ID": "encounter_id",
    "DH_ORDER_ID": "order_id",
})

# --- Dtypes & basic cleanup ---
for c in ["patient_id", "encounter_id", "order_id"]:
    med[c] = med[c].astype("string[pyarrow]")

# drop null keys and duplicate orders within the same encounter
med = (
    med
    .dropna(subset=["patient_id", "encounter_id", "order_id"])
    .drop_duplicates(subset=["patient_id", "encounter_id", "order_id"])
)

# --- Aggregate unique order count per encounter ---
med_enc = (
    med.groupby(["patient_id", "encounter_id"], as_index=False)
       .agg(med_order_nunique=("order_id", lambda s: s.dropna().nunique()))
)

med_enc["med_order_nunique"] = med_enc["med_order_nunique"].astype("Int64")
for c in ["patient_id", "encounter_id"]:
    med_enc[c] = med_enc[c].astype("string[pyarrow]")

# --- Merge into official store ---
patient_features = patient_features.merge(med_enc, on=["patient_id", "encounter_id"], how="left")

print(patient_features.head(5))

                                          patient_id  \
0  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
1  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
2  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
3  00010b29ec5fd2ad40834318f6b429961059a614fbfece...   
4  000183804ffd1929b01d145a9cbf729d9c411d64675c02...   

                                        encounter_id  \
0  0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...   
1  b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...   
2  c67b56e4f422be9a8894634536e8895da82d159b2ed583...   
3  0d79127152552683ecf7ba4acd391eeaaa6e90cceffb39...   
4  affeaca0615c02f762a8404e381b3b86243ee1b7e42bf2...   

                              icd_prefix3_unique_str  n_icd_prefix3  \
0                            E11,E86,F31,I50,J44,N30              6   
1  E11,E78,E86,F17,F31,F41,F43,G70,I50,J44,K21,N3...             15   
2                                            E87,R53              2   
3                                         

In [8]:
proc_path = f"{BASE}/grai_matter_fct_procedure_codes/"
proc_cols = [
    "PRIMARY_PATIENT_IDENTIFIER",
    "DH_ENCOUNTER_ID",
    "PROCEDURE_CODE",
    "PROCEDURE_CODE_DESCRIPTION",
    "PROCEDURE_DT",
]

try:
    proc = pd.read_parquet(proc_path, columns=proc_cols, dtype_backend="pyarrow")
except Exception:
    fs = s3fs.S3FileSystem()
    proc = pd.read_parquet(proc_path, columns=proc_cols, engine="pyarrow", filesystem=fs)

# --- Standardize column names ---
proc = proc.rename(columns={
    "PRIMARY_PATIENT_IDENTIFIER": "patient_id",
    "DH_ENCOUNTER_ID": "encounter_id",
    "PROCEDURE_CODE": "procedure_code",
    "PROCEDURE_CODE_DESCRIPTION": "procedure_desc",
    "PROCEDURE_DT": "procedure_dt",
})

# --- Dtypes & basic cleanup ---
for c in ["patient_id", "encounter_id", "procedure_code", "procedure_desc"]:
    proc[c] = proc[c].astype("string[pyarrow]")
proc["procedure_dt"] = pd.to_datetime(proc["procedure_dt"], errors="coerce")

# keep rows with complete keys and datetime; de-dupe exact duplicates
proc = (
    proc
    .dropna(subset=["patient_id", "encounter_id", "procedure_dt"])
    .drop_duplicates(subset=["patient_id", "encounter_id", "procedure_code", "procedure_desc", "procedure_dt"])
)

# light normalization for stable sequences (idempotent)
proc["procedure_code"] = proc["procedure_code"].str.strip().str.upper()
proc["procedure_desc"] = proc["procedure_desc"].str.strip()

# --- Order rows globally by key and time to ensure aligned sequences ---
proc_sorted = proc.sort_values(["patient_id", "encounter_id", "procedure_dt"])

# --- Helpers for NA-safe joins ---
def _join_str(s: pd.Series) -> str:
    return "|".join(s.astype("string[pyarrow]").fillna("").astype(str))

def _join_time(s: pd.Series) -> str:
    return "|".join(s.dt.strftime("%Y-%m-%d %H:%M:%S"))

# --- Build aligned sequences per encounter and total count ---
proc_seq = (
    proc_sorted
    .groupby(["patient_id", "encounter_id"], as_index=False)
    .agg(
        proc_codes_seq_str=("procedure_code", _join_str),
        proc_descs_seq_str=("procedure_desc", _join_str),
        proc_times_seq_str=("procedure_dt",   _join_time),
        proc_count=("procedure_code", "size"),
    )
)

# enforce dtypes for outputs
for c in ["patient_id", "encounter_id", "proc_codes_seq_str", "proc_descs_seq_str", "proc_times_seq_str"]:
    proc_seq[c] = proc_seq[c].astype("string[pyarrow]")
proc_seq["proc_count"] = proc_seq["proc_count"].astype("Int64")

# --- Merge into the official store (encounter-level) ---
patient_features = patient_features.merge(
    proc_seq, on=["patient_id", "encounter_id"], how="left"
)

print(patient_features.head(5))

                                          patient_id  \
0  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
1  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
2  0000fa8c4cfecf9f7e26e462a990af1ad85a1bfab7f43e...   
3  00010b29ec5fd2ad40834318f6b429961059a614fbfece...   
4  000183804ffd1929b01d145a9cbf729d9c411d64675c02...   

                                        encounter_id  \
0  0cf2b96aa588db64f96239a66b9d8c640489f1959aa678...   
1  b435c02020a0f4caad1e81c7947504f0483d4b4d2bf15a...   
2  c67b56e4f422be9a8894634536e8895da82d159b2ed583...   
3  0d79127152552683ecf7ba4acd391eeaaa6e90cceffb39...   
4  affeaca0615c02f762a8404e381b3b86243ee1b7e42bf2...   

                              icd_prefix3_unique_str  n_icd_prefix3  \
0                            E11,E86,F31,I50,J44,N30              6   
1  E11,E78,E86,F17,F31,F41,F43,G70,I50,J44,K21,N3...             15   
2                                            E87,R53              2   
3                                         

In [9]:
patient_features.to_csv("patient_features_new.csv", index=False)
print("Saved: patient_features_new.csv")

Saved: patient_features_new.csv
